# Step 1: Exploratory Data Analysis (EDA) & Missing Value Imputation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# 1. Load the dataset (assuming the file is named 'water_potability.csv')
df = pd.read_csv('water_potability.csv')

# Check for missing values before cleaning
print("Missing values before cleaning:")
print(df.isnull().sum())

In [ ]:
# 2. Impute missing values using the median grouped by the target class (Potability)
# This preserves the underlying distribution characteristics for safe vs. unsafe water
for col in ['ph', 'Sulfate', 'Trihalomethanes']:
    df[col] = df.groupby('Potability')[col].transform(lambda x: x.fillna(x.median()))

print("\nMissing values after cleaning:", df.isnull().sum().sum())

In [ ]:
# 3. Visualize feature correlations (required for the Final Report and reduction analysis)
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

# Step 2: Data Splitting & Baseline Model Implementation

In [ ]:
# Separate features (X) and target variable (y)
X = df.drop('Potability', axis=1)
y = df['Potability']

# Split the dataset into training and testing sets (80% / 20%)
# Stratify ensures the class proportions are maintained in both subsets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature Scaling (Standardization)
# Essential because features operate on vastly different scales and units
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and train the baseline MLP model
# Setting a simple architecture (one hidden layer with 64 neurons) as a benchmark
baseline_mlp = MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42)
baseline_mlp.fit(X_train_scaled, y_train)

# Prediction and baseline evaluation
y_pred_base = baseline_mlp.predict(X_test_scaled)

print("=== BASELINE MODEL RESULTS (FULL FEATURE SET) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_base):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_base))

# Plot baseline Confusion Matrix
sns.heatmap(confusion_matrix(y_test, y_pred_base), annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - Baseline Model")
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# Step 3: Feature Space Reduction

In [ ]:
# Determine feature importance using a Random Forest auxiliary model
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

# Visualize feature importances 
plt.figure(figsize=(8, 5))
sns.barplot(x=importances.values, y=importances.index, palette="viridis")
plt.title("Feature Importances in Water Potability Dataset")
plt.xlabel("Importance Score")
plt.show()

print("Feature importances sorted:")
print(importances)

# REDUCTION: Drop low-impact features to test dimensionality reduction hypothesis
# Selecting only the top 6 most critical features
top_features = importances.index[:6].tolist()
print("\nSelected features for the reduced model:", top_features)

X_train_red = pd.DataFrame(X_train, columns=X.columns)[top_features]
X_test_red = pd.DataFrame(X_test, columns=X.columns)[top_features]

# Re-apply standardization to the reduced feature space
scaler_red = StandardScaler()
X_train_red_scaled = scaler_red.fit_transform(X_train_red)
X_test_red_scaled = scaler_red.transform(X_test_red)

# Train the baseline architecture on the reduced dataset
mlp_reduced = MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42)
mlp_reduced.fit(X_train_red_scaled, y_train)

y_pred_red = mlp_reduced.predict(X_test_red_scaled)
print("\n=== BASELINE MODEL RESULTS (REDUCED FEATURE SET) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_red):.4f}")

# Step 4: Iterative Hyperparameter Optimization (Grid Search)

In [ ]:
# --- Round 1: Tuning Network Architecture and Activation Functions ---
param_grid_r1 = {
    'hidden_layer_sizes': [(32, 16), (64, 32), (100,)],
    'activation': ['tanh', 'relu'],
    'solver': ['adam'],
    'max_iter': [600]
}

grid_r1 = GridSearchCV(MLPClassifier(random_state=42), param_grid_r1, cv=3, scoring='accuracy', n_jobs=-1)
grid_r1.fit(X_train_red_scaled, y_train)

print("Best Parameters - Round 1:")
print(grid_r1.best_params_)

# --- Round 2: Fine-Tuning Learning Rate and L2 Regularization (Alpha) ---
# Reusing the winning configurations from Round 1 dynamically
param_grid_r2 = {
    'hidden_layer_sizes': [grid_r1.best_params_['hidden_layer_sizes']],
    'activation': [grid_r1.best_params_['activation']],
    'learning_rate_init': [0.001, 0.01, 0.1],
    'alpha': [0.0001, 0.001, 0.01],
    'max_iter': [600]
}

grid_r2 = GridSearchCV(MLPClassifier(random_state=42), param_grid_r2, cv=3, scoring='accuracy', n_jobs=-1)
grid_r2.fit(X_train_red_scaled, y_train)

print("\nBest Parameters - Round 2 (Final Optimized Model):")
print(grid_r2.best_params_)

# Step 5: Final Evaluation Report

In [ ]:
# Extract and evaluate the final optimized model on the test set 
best_model = grid_r2.best_estimator_
y_pred_final = best_model.predict(X_test_red_scaled)

print("\n=== FINAL OPTIMIZED MODEL RESULTS ===")
print(f"Final Accuracy: {accuracy_score(y_test, y_pred_final):.4f}")
print("\nFinal Classification Report:")
print(classification_report(y_test, y_pred_final))

# Plot comparative confusion matrices side-by-side for the final report 
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Baseline model confusion matrix
sns.heatmap(confusion_matrix(y_test, y_pred_base), annot=True, fmt='d', cmap='Oranges', ax=ax[0])
ax[0].set_title(f"Baseline Model (All Features)\nAccuracy: {accuracy_score(y_test, y_pred_base):.3f}")
ax[0].set_ylabel('Actual')
ax[0].set_xlabel('Predicted')

# Plot 2: Optimized model confusion matrix
sns.heatmap(confusion_matrix(y_test, y_pred_final), annot=True, fmt='d', cmap='Greens', ax=ax[1])
ax[1].set_title(f"Optimized Model (Reduced Features + Tuned)\nAccuracy: {accuracy_score(y_test, y_pred_final):.3f}")
ax[1].set_ylabel('Actual')
ax[1].set_xlabel('Predicted')

plt.tight_layout()
plt.show()